<!-- PROVENANCE: FORK-ORIGINAL — fork-only file (RA timeseries project); not in upstream earthrise-media/mining-detector -->
# Explore monthly-vs-quarterly pilot results
Thin interactive layer over the outputs of `analysis.py` (run that first — or let `run_overnight.sh` run it).
All heavy computation lives in `analysis.py`; this notebook only loads its outputs plus the raw panels for ad-hoc digging.

In [ ]:
from pathlib import Path
import pandas as pd, geopandas as gpd, numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'timeseries' else Path.cwd()
TS = ROOT / 'data/timeseries'
panels = sorted((TS/'panels').glob('panel_*.parquet'))
print(f'{len(panels)} panels found')
df = gpd.GeoDataFrame(pd.concat([gpd.read_parquet(p) for p in panels], ignore_index=True))
df.head()

## The generated report (rendered inline)

In [ ]:
from IPython.display import Markdown
rep = TS/'analysis/REPORT.md'
Markdown(rep.read_text().replace('figs/', str(TS/'analysis/figs')+'/')) if rep.exists() else 'Run analysis.py first.'

## Tables produced by the analysis

In [ ]:
for t in sorted((TS/'analysis/tables').glob('*.csv')):
    print('='*80); print(t.name)
    display(pd.read_csv(t))

## Ad-hoc: pick a chip, plot its monthly trajectory with the cloud flag
Grey band = months where the chip was not fully observed (clear_frac < 0.95) — exactly the observations an interpolation model should down-weight or gate.

In [ ]:
m = df[df.cadence=='monthly'].copy()
# example: the chip with the highest mean prob (an active mine); swap in any chip_id
chip = m.groupby('chip_id')['prob'].mean().idxmax()
sub = m[m.chip_id==chip].sort_values('period')
fig, ax = plt.subplots(figsize=(9,3.5))
ax.plot(sub.period, sub.prob, 'o-', label='CNN prob')
for _, r in sub.iterrows():
    if r.clear_frac < 0.95:
        ax.axvspan(r.period, r.period, color='grey', alpha=.4, lw=8)
ax.axhline(0.4, color='r', ls='--', lw=.8, label='t=0.4')
ax.set_title(f'{chip}'); ax.tick_params(axis='x', rotation=45); ax.legend(); plt.tight_layout()

## Ad-hoc: map of one period's probabilities

In [ ]:
period = 'm2025-07'  # dry season; change at will
sub = df[df.period==period]
if len(sub):
    ax = sub.plot(column='prob', markersize=1, figsize=(10,7), legend=True, cmap='viridis')
    sub[sub.clear_frac<0.95].boundary.plot(ax=ax, color='red', lw=.2)
    ax.set_title(f'{period}: prob (red outline = cloud-flagged chips)')